In [10]:
from pydub import AudioSegment
from openai import OpenAI
import json

In [2]:
interview = AudioSegment.from_file('../assets/025_lei_chou.wav')
ten_minutes = 10 * 60 * 1000
first_10_minutes = interview[:ten_minutes]
first_10_minutes.export('../assets/025_lei_chou_10min.mp3', format='mp3')

<_io.BufferedRandom name='../assets/025_lei_chou_10min.mp3'>

In [3]:
client = OpenAI()
audio_file = open('../assets/025_lei_chou_10min.mp3', 'rb')

In [7]:
transcript = client.audio.transcriptions.create(
  file=audio_file,
  model='whisper-1',
  response_format='verbose_json',
  timestamp_granularities=['word'],
  language='en'
)

In [12]:
json.dump(dict(transcript), open('../assets/025_lei_chou_10min_transcript.json', 'w'), indent=2)

{'text': "Okay, ready? So we start, if you could just say your name, how old you are, today's date, and where we are. My name is Lee Chow, I'm 36, and today is May 5th, 2003. And where are we? We're in New York City in my apartment. In the lovely East Village. In the East Village. Okay. So where were you born, Lee? I was born in Taiwan. When did you come to New York? I came to New York in the winter of 84. Okay. And what did you come here to do? For college. Where did you go to school? Cooper Union. So were you already out as a gay man when you came to Cooper? When I moved to New York, I came out to my family. So when you were living the life of the gay arts student at Cooper Union, and what was that like in 84? It was a very interesting experience, actually. It was fun. What were you studying? Sculpture. And were there a lot of openly gay people at school? No. No. There was a few that I found out after we graduated. There wasn't any, wasn't a very visible gay presence there. So the id

In [19]:
system_prompt = """
    You are a helpful assistant from the ACT UP Oral History Project. You are
    helping to transcribe an interview with an activist. They are discussing they
    work with the AIDS Coalition to Unleash Power (ACT UP) in New York City in the
    1980s and 1990s. Be sure to correct misspellings, including the terms "AIDS
    Coalition to Unleash Power" and "ACT UP," "AZT," "AIDS," and "GMHC."
"""

response = client.chat.completions.create(
    model='gpt-4o',
    messages=[
        {
            'role': 'system',
            'content': system_prompt
        },
        {
            'role': 'user',
            'content': str(transcript.words),
        },
    ],
)

In [18]:
dict(response.choices[0].message)
json.dump(dict(response.choices[0].message), open('../assets/025_lei_chou_10min_transcript_corrected.json', 'w'), indent=2)